In [0]:
gold_df = spark.table("workspace.default.silver_sales")

In [0]:
gold_df.show(5)

gold_df.printSchema()

print("Total Records :", gold_df.count())

In [0]:
from pyspark.sql.functions import sum

gold_df.select(
    sum("total_amount").alias("Total_Revenue")
).show()

In [0]:
from pyspark.sql.functions import count

gold_df.select(
    count("order_id").alias("Total_Orders")
).show()

In [0]:
from pyspark.sql.functions import avg

gold_df.select(
    avg("total_amount").alias("Average_Order_Value")
).show()

In [0]:
from pyspark.sql.functions import max

gold_df.select(
    max("total_amount").alias("Highest_Revenue")
).show()

In [0]:
from pyspark.sql.functions import min 

gold_df.select(
    min("total_amount").alias("Highest_Revenue")
).show()

In [0]:
from pyspark.sql.functions import sum

category_sales_df=(
    gold_df
    .groupBy("Category")
    .agg(
        sum("total_amount").alias("total_revenue")
    )
)

category_sales_df.show()

In [0]:
category_sales_df.orderBy(
    "total_revenue",
    ascending=False
).show()

In [0]:
from pyspark.sql.functions import col
city_sales_df = (
    gold_df
    .groupBy("city")
    .agg(
        sum("total_amount").alias("city_revenue")
    )
)

city_sales_df.orderBy(
    col("city_revenue").desc()
).show()

In [0]:
category_sales_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_category_summary")

In [0]:
city_sales_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.gold_city_summary")

In [0]:
from pyspark.sql.functions import sum

customer_revenue_df = (
    gold_df
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        sum("total_amount").alias("customer_revenue")
    )
)

customer_revenue_df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
window_spec = Window.orderBy(
    col("customer_revenue").desc()
)

In [0]:
top_customer_df = customer_revenue_df.withColumn(
    "rank",
    row_number().over(window_spec)
)

In [0]:
top_customer_df.filter(
    col("rank") <= 10
).show()

In [0]:
top_customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_customers")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

In [0]:
sales_df = sales_df.withColumn(
    "next_order_date",
    lead("order_date", 1).over(window_spec)
)